In [ ]:
# # Uncomment on Colab

# !pip install exponax chaotax

This notebook will generate the training and testing data for the second notebook. 

We will run same simulator on a finer resolution and then use `ex.map_between_resolutions` to interpolate the data to the coarser resolution. Running this on a Colab GPU T4 should take 10-15 minutes. Prefer downloading the data from Huggingface: https://huggingface.co/datasets/ceyron/jax-ad-workshop

In [1]:
import jax
import jax.numpy as jnp
import exponax as ex
import chaotax

In [2]:
DOMAIN_EXTENT = 2 * jnp.pi
DT = 0.1
NUM_SUBSTEPS = 10
DIFFUSIVITY = 1e-2
INTEGRATION_ORDER = 4

N_full = 128
N_coarse = 32

NUM_SAMPLES_TRAIN = 200
NUM_SAMPLES_TEST = 20

NUM_TIME_STEPS_TRAIN = 100
NUM_TIME_STEPS_TEST = 300

NUM_WARMUP_STEPS = 1000

In [3]:
stepper_fine = ex.RepeatedStepper(
    ex.stepper.KolmogorovFlowVorticity(
        2,
        DOMAIN_EXTENT,
        N_full,
        DT / NUM_SUBSTEPS,
        diffusivity=DIFFUSIVITY,
        order=INTEGRATION_ORDER,
    ),
    NUM_SUBSTEPS,
)

In [4]:
train_ic_set_full = jax.random.normal(
    jax.random.PRNGKey(0),
    (NUM_SAMPLES_TRAIN, 1, N_full, N_full),
)

In [5]:
train_ic_set_warmed_full = jax.vmap(
    ex.repeat(
        stepper_fine,
        NUM_WARMUP_STEPS,
    )
)(train_ic_set_full)

In [6]:
lyapunov_exponent = chaotax.lyapunov(
    stepper_fine,
    rollout_steps=2000,
    warmup_steps=1000,
    discard_steps=500,
)(
    jax.random.normal(
        jax.random.PRNGKey(1),
        (1, N_full, N_full),
    ),
    jax.random.normal(
        jax.random.PRNGKey(2),
        (1, N_full, N_full),
    ),
)

In [7]:
lyapunov_exponent, 1 / lyapunov_exponent

(Array(0.03791158, dtype=float32), Array(26.377163, dtype=float32))

In [8]:
train_trj_set_full = jax.vmap(
    ex.rollout(stepper_fine, NUM_TIME_STEPS_TRAIN, include_init=True)
)(train_ic_set_warmed_full)

In [9]:
train_trj_set_coarsened = jax.vmap(
    jax.vmap(lambda state: ex.map_between_resolutions(state, N_coarse))
)(train_trj_set_full)

In [10]:
test_ic_set_full = jax.random.normal(
    jax.random.key(1),
    (NUM_SAMPLES_TEST, 1, N_full, N_full),
)

In [11]:
test_ic_set_warmed_full = jax.vmap(
    ex.repeat(
        stepper_fine,
        NUM_WARMUP_STEPS,
    )
)(test_ic_set_full)

In [12]:
test_trj_set_full = jax.vmap(
    ex.rollout(stepper_fine, NUM_TIME_STEPS_TEST, include_init=True)
)(test_ic_set_warmed_full)

In [13]:
test_ic_set_warmed_coarsened = jax.vmap(
    lambda state: ex.map_between_resolutions(state, N_coarse),
)(test_ic_set_warmed_full)

In [14]:
test_trj_set_coarsened = jax.vmap(
    jax.vmap(lambda state: ex.map_between_resolutions(state, N_coarse))
)(test_trj_set_full)

In [15]:
jnp.save("train_trj_set_coarsened.npy", train_trj_set_coarsened.astype(jnp.float16))
jnp.save("test_trj_set_coarsened.npy", test_trj_set_coarsened.astype(jnp.float16))